# ETL y datasets derivados

## Objetivo

Explicar como los datos raw se transforman en datasets analiticos para
indicadores, modelado, clustering, reglas de asociacion y regresion agregada.

La fase conserva trazabilidad: los desasignados permanecen en el dataset
maestro, pero se excluyen del dataset de clasificacion porque no representan un
resultado academico evaluado comparable.


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

OUTPUTS = ROOT / "outputs"
TABLES = OUTPUTS / "tables"
FIGURES = OUTPUTS / "figures"
REPORTS = OUTPUTS / "reports"

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)


## Datasets derivados generados

| dataset | archivo | filas | columnas | conclusion |
| --- | --- | --- | --- | --- |
| ds_estudiante_curso_resultado_2024 | data/processed/ds_estudiante_curso_resultado_2024.csv | 2879 | 58 | Unidad estudiante-curso-periodo construida desde asignaciones y actas. |
| ds_resumen_curso_periodo_2024 | data/processed/ds_resumen_curso_periodo_2024.csv | 132 | 42 | Unidad curso-periodo lista para indicadores y regresion agregada. |
| ds_continuidad_oferta_2024 | data/processed/ds_continuidad_oferta_2024.csv | 90 | 24 | Unidad curso anual lista para analizar continuidad de oferta. |
| ds_cursos_criticos_clustering | data/processed/ds_cursos_criticos_clustering.csv | 90 | 15 | Dataset numerico y contextual preparado para clustering. |
| ds_modelo_clasificacion | data/processed/ds_modelo_clasificacion.csv | 2726 | 21 | Dataset de modelado sin desasignados ni variables prohibidas por fuga. |
| ds_reglas_asociacion | data/processed/ds_reglas_asociacion.csv | 2726 | 44 | Matriz transaccional codificada para reglas de asociacion. |

Interpretacion: cada dataset derivado tiene una unidad analitica distinta. Esto
evita mezclar niveles de analisis: estudiante-curso-periodo, curso-periodo,
curso anual, transaccion o registro modelable.

Conclusion: la arquitectura de datos permite analizar el problema desde varios
angulos sin duplicar reglas de negocio en notebooks.


In [ ]:
datasets_derivados = pd.read_csv(TABLES / "04_resumen_datasets_derivados.csv")
datasets_derivados


## Distribucion de targets

| target | valor | conteo | porcentaje |
| --- | --- | --- | --- |
| necesito_recuperacion | False | 2147 | 78.76% |
| necesito_recuperacion | True | 579 | 21.24% |
| perdida_definitiva | False | 1899 | 69.66% |
| perdida_definitiva | True | 827 | 30.34% |
| riesgo_retraso_potencial | False | 2315 | 84.92% |
| riesgo_retraso_potencial | True | 411 | 15.08% |
| riesgo_retraso_potencial_amplio | False | 2207 | 80.96% |
| riesgo_retraso_potencial_amplio | True | 519 | 19.04% |

Interpretacion: `perdida_definitiva` representa que el ultimo resultado
observable no fue aprobado; no significa necesariamente ausencia de recuperacion.
Los targets de riesgo combinan perdida observada con continuidad de oferta y
condicion curricular, por lo que deben leerse como riesgo potencial.

Conclusion: las clases positivas no tienen la misma prevalencia. Por eso las
fases de modelado usan metricas como precision, recall, F1 y ROC-AUC, no solo
accuracy.


In [ ]:
targets = pd.read_csv(TABLES / "04_distribucion_targets.csv")
targets


## Decisiones metodologicas relevantes

- El dataset maestro parte de asignaciones para conservar desasignaciones y
  cruzar resultados de actas cuando existen.
- Los desasignados se preservan para trazabilidad, pero se excluyen del dataset
  de clasificacion.
- La columna de docencia no se utiliza como variable analitica ni de modelado.
- Las variables con fuga de informacion, como notas finales y resultado final,
  se excluyen de los predictores.
- Los datasets 02, 03 y 09 se describen como provistos por el Departamento de
  Computo.

## Hallazgos obtenidos

- Se generaron seis datasets procesados principales para cubrir indicadores,
  clasificacion, clustering y reglas de asociacion.
- El dataset de clasificacion queda con 2726 registros modelables.
- La perdida definitiva observada aparece en 827 registros modelables, que
  equivalen a 30.34% del dataset de clasificacion.
- El riesgo de retraso academico potencial aparece en 411 registros, equivalente
  a 15.08%.

## Conclusiones del notebook

La fase de ETL convierte archivos institucionales en datasets analiticos con
unidades claras y decisiones explicitas. Esto reduce ambiguedad al interpretar
indicadores, pruebas estadisticas y modelos.

## Conclusiones generales

El proyecto cuenta con una base procesada reproducible. Las variables objetivo
son utiles para priorizar patrones academicos, pero no deben interpretarse como
diagnosticos individuales ni como evidencia causal.
